In [12]:
#!pip install numpy 
#!pip install pandas 
#!pip install matplotlib
#!pip install librosa
#!pip install scipy
#!pip install sklearn 
#!pip install torch 
##!pip install torch.nn
#!pip install torch.utils
#!pip install pyro-ppl

import numpy as np
# For reproducibility
np.random.seed(33)
import pandas as pd
import matplotlib.pyplot as plt

import librosa                                     # To manage the audio files
import librosa.display
import librosa.feature

import os

import pickle

from scipy.io import wavfile as wav

from sklearn.preprocessing import LabelEncoder, StandardScaler
# from sklearn.model_selection import train_test_split 

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split
from torch.utils.data import Dataset, DataLoader

import pyro
from pyro.distributions import Normal, Categorical
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import Adam

# Setting the input and output folders
# implementare funzione per prendere solo input e produrre cartella di output rinominandola come l'input ed eliminando la cartella in input
input_dataset = './Chicks_Automatic_Detection_dataset_2/Registrazioni/'
output_dataset = './Chicks_Automatic_Detection_dataset_2/dt_Registrazioni_modificate/'
# output_dataset = './Chicks_Automatic_Detection_dataset_2/Registrazioni_prova/'

# Some functions definition
Two functions have been defined. The first used to save the model after every epoch and the second to load the last fully computed model in case the training has been interrupted at some point

In [2]:
# Function to save the model 
def save_checkpoint(model, optimizer, epoch, loss, accuracy, filename='bnn_checkpoint.pth'):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'pyro_params': {name: pyro.param(name).detach().cpu() for name in pyro.get_param_store().get_all_param_names()},
        'loss': loss,
        'accuracy': accuracy,
    }
    torch.save(checkpoint, filename)
    print(f"Checkpoint saved at epoch {epoch} with loss {loss:.4f} and accuracy {accuracy:.4f}")

# Function to load the model
def load_checkpoint(model, optimizer, filename='bnn_checkpoint.pth'):
    if os.path.isfile(filename):
        checkpoint = torch.load(filename)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        # Load Pyro parameters
        pyro_param_store = pyro.get_param_store()
        for name, param in checkpoint['pyro_params'].items(): 
            pyro_param_store[name] = param

        start_epoch = checkpoint['epoch'] + 1
        loss = checkpoint['loss']
        accuracy = checkpoint['accuracy']
        print(f"Checkpoint loaded. Resuming training from epoch {start_epoch} with loss {loss:.4f} and accuracy {accuracy:.4f}")
        return start_epoch, loss, accuracy
    else:
        print("No checkpoint found. Starting training from scratch.")
        return 0, None, None

# Preprocessing step
- creo un secondo dataset (visto che non sono sicura vorrei evitare di sputtanarlo e poi doverlo riscaricare e perdere altro tempo)
- nel nuovo dt metto solo i file audio ('.wav') che sono quelli che effettivamente mi serviranno per fare il training
-  Per ogni audio tolgo i periodi di silenzo e ne faccio una segmentazione dinamica. Computo per ogni segmento lo spettrogramma, ne riduco la dimensionalità ad un array 1D ( dove ogni valore rappresenta una coppia frequenza-ampiezza specifica per uno t specifico) e lo normalizzo. Il tutto, assime ad una specifica etichetta (riporta il tipo di 'call' per la categorizzazione), viene salvato in un file '.pkl' all'interno di una cartella specifica
- Gli spettrogrammi ottenuti sono di dimensioni diverse e, per essere usati come imput all'interno di una rete, devono avere tutti la stessa dimensione. La soluzione a cui sono giunta 
è, dopo aver trovato lo spettrogramma più "grande", riempire con degli zeri posti all'inizio tutti gli altri di modo da renderli tutti con le stesse dimensioni. I risultati vengono salvati sovrascrivendo gli spettrogrammi computati al punto precedente.

Note:
- in caso il progetto raggiungesse una fase più "seria", la politica di gestione del database di partenza dovrebbe sicuramente essere modificata e automatizzata. In questo senso bisognerà modificare anche tutta la fase di preprocessing di modo da applicarla solo ai nuovi dati e non all'intero dataset tutte le volte
- Facendo una segmentazione dinamica c'è il rischio di over-segmentation per quanto riguarda le contact calls... Come si potrebbe mitigare?
- sia per le high noise pleasure calls che per le low noise pleasure calls sarebbe opportuno prima eliminare il rumore di sottofondo e poi avviarle alla fase di preprocessing?
- questa fase di preprocessing è davvero lenta e richiede molte risorse, c'è forse un modo per migliorare anche in vista di un (forse) futuro deployment per essere effettivamente usata?

Da aggiungere:
- metodo per salvare il valore della variabile "input_size" (size del più grande tra gli spettrogrammi, che verrà usata come parametro per la rete) di modo da non dover tutte le volte lanciare il preprocessing (probabilmente utilizzerò un .json)

In [11]:
# Change the file name and location
def file_name(input_dataset, output_dataset):
    for subdir, dirs, files in os.walk(input_dataset):
        for file in files:
            if file.endswith('.wav'):
                # Current file path
                current_file_path = os.path.join(subdir, file)
                # Extract the relevant subdirectories to form part of the new file name
                relative_subdir = os.path.relpath(subdir, input_dataset)
                relative_subdir = relative_subdir.replace(os.sep, '_')  # Replace directory separators with underscores
                
                # Form the new file name
                new_file_name = f"{relative_subdir}_{file}"
                
                # New file path in the output directory with the new name
                new_file_path = os.path.join(output_dataset, new_file_name)
                
                # Create the output directory if it doesn't exist
                os.makedirs(output_dataset, exist_ok=True)
                
                # Move and rename the file
                os.rename(current_file_path, new_file_path)
                print(f"Moved and renamed: {current_file_path} -> {new_file_path}")


def compute_spectrogram(y, sr):
    # n_fft determines the window size over which the Fourier Transform is computed
    # Adjust n_fft based on segment length
    n_fft = min(2048, len(y))
    hop_length = n_fft // 2  # Typically, hop_length is half of n_fft
    spectrogram = librosa.stft(y, n_fft=n_fft, hop_length=hop_length)
    spectrogram = librosa.amplitude_to_db(np.abs(spectrogram), ref=np.max)

    # # Plot the spectrogram
    # plt.figure(figsize=(10, 4))
    # librosa.display.specshow(spectrogram, sr=sr, hop_length=hop_length, x_axis='time', y_axis='log')
    # plt.colorbar(format='%+2.0f dB')
    # plt.title('Spectrogram')
    # plt.tight_layout()
    # plt.show()

    return spectrogram

def audio_spectrograms(output_dataset, subdirectory='audio_spectrograms'):
    save_dir = os.path.join(output_dataset, subdirectory) # Where the segments spectrograms will be saved
    # scaler = StandardScaler() # For spectrograms normalization

    # Get a list of all audio files in the directory
    audio_files = [os.path.join(output_dataset, f) for f in os.listdir(output_dataset) if f.endswith('.wav')]

    ########## Steps ##########
    # 1. Silences removal at the beginning and at the end 
    # 2. Division of the audio in 20sec long segments
    # 3. Creation of a spectrogram for each segment

    #Loop through all the files
    for audio_file in audio_files:
        # Load the audio file
        y, sampling_rate = librosa.load(audio_file, sr=None)

        # Step n°1
        # It's either I choose a dynamic trimming process...
        # rmse = librosa.feature.rms(y=y, frame_length=256, hop_length=64)[0]

        # Debug: print the RMSE values and check their range
        #print("RMSE values (first 10):", rmse[:10])
        
        #specs = librosa.power_to_db(rmse**2, ref=np.max) 

        # top_db = int(min(specs)) - 2
        # ...or a static one
        ytrim, _ = librosa.effects.trim(y, frame_length=256, hop_length=64, top_db=55)

        trimmed_duration = librosa.get_duration(y=ytrim, sr=sampling_rate)

        # Step n°2
        # Extract the base name of the audio file (without extension)
        base_name = os.path.splitext(os.path.basename(audio_file))[0]

        chunk_duration = 20 # Every chunck is 20sec long
        # Get number of samples for 20 seconds
        buffer = int(chunk_duration * sampling_rate)

        audio_length = len(ytrim)
        audio_done = 0 # Tracks the alredy prcessed portion of the audio
        counter = 1

        while audio_done < audio_length:
            # Check if the chunck duration is actually contained inside the audio
            if buffer > (audio_length - audio_done):
                buffer = audio_length - audio_done

            # Check the duration of the current chunk
            chunk_duration_sec = librosa.get_duration(y=ytrim[audio_done : (audio_done + buffer)], sr=sampling_rate)
            # Skip chunks less than 20 seconds
            if chunk_duration_sec < 20:
                break  # Exit the loop if no more valid chunks

            segments = ytrim[audio_done : (audio_done + buffer)]

            # Step n°3
            segment_spectrogram = compute_spectrogram(segments, sampling_rate)

            # Saving the spectrogram with its corresponding lable
            segment_spectrogram_name = f"{base_name}_segment_{counter}.pt"
            segment_spectrogram_path = os.path.join(save_dir, segment_spectrogram_name)
            # Extracting the type of call
            call_type = base_name.split('_')[0]

            # Save the normalized spectrogram and its corresponding label as a Torch tensor
            data = {
                'spectrogram': torch.tensor(segment_spectrogram, dtype=torch.float32),
                'label': call_type
            }
            torch.save(data, segment_spectrogram_path)

            # Save the spectrogram as an image (.png)
            image_file_name = f"{base_name}_segment_{counter}.png"
            image_file_path = os.path.join(save_dir, image_file_name)

            plt.figure(figsize=(10, 4))
            librosa.display.specshow(segment_spectrogram, sr=sampling_rate, hop_length=int(buffer / 2), x_axis='time', y_axis='log', cmap='viridis')
            plt.colorbar(format='%+2.0f dB')
            plt.title(f'Spectrogram of {base_name}_segment_{counter}')
            plt.tight_layout()
            plt.savefig(image_file_path)  # Save the spectrogram as an image file
            plt.close()  # Close the plot to free up memory
            
            counter += 1
            audio_done += buffer # Update the position in the audio

print ("starting")

#file_name(input_dataset, output_dataset) # To reorder the dataset

audio_spectrograms(output_dataset)

print("ending")


starting


FileNotFoundError: [Errno 2] No such file or directory: './Chicks_Automatic_Detection_dataset_2/Registrazioni_modificate/'

# BNN Architecture
Following there is the implementation of a simple BNN with an input layer, 2 hidden layers and an output layer

In [6]:
# Bayesian Neural Network implementation
print("model definition")
class BNN(nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super (BNN, self).__init__()
    self.fc1 = nn.Linear(input_size, hidden_size)
    self.fc2 = nn.Linear(hidden_size, hidden_size)
    self.out = nn.Linear(hidden_size, output_size)

  def forward(self, x):
    output = F.relu(self.fc1(x))
    output = F.relu(self.fc2(x))
    output = self.out(output)
    return output

  def model(self, x_data, y_data = None):
    #Prior distributions for weightes and biases
    fc1w_prior = Normal(loc=torch.zeros_like(self.fc1.weight), scale=torch.ones_like(self.fc1.weight)).to_event(self.fc1.weight.dim())
    fc1b_prior = Normal(loc=torch.zeros_like(self.fc1.bias), scale=torch.ones_like(self.fc1.bias)).to_event(self.fc1.bias.dim())

    fc2w_prior = Normal(loc=torch.zeros_like(self.fc2.weight), scale=torch.ones_like(self.fc2.weight)).to_event(self.fc2.weight.dim())
    fc2b_prior = Normal(loc=torch.zeros_like(self.fc2.bias), scale=torch.ones_like(self.fc2.bias)).to_event(self.fc2.bias.dim())

    outputw_prior = Normal(loc=torch.zeros_like(self.out.weight), scale=torch.ones_like(self.out.weight)).to_event(self.out.weight.dim())
    outputb_prior = Normal(loc=torch.zeros_like(self.out.bias), scale=torch.ones_like(self.out.bias)).to_event(self.out.bias.dim())

    priors = {'fc1.weight': fc1w_prior, 'fc1.bias': fc1b_prior, 'fc2.weight': fc2w_prior, 'fc2.bias': fc2b_prior, 'output.weight': outputw_prior, 'output.bias': outputb_prior}
    
    # Lift module parameters to random variables sampled from the priors
    lifted_module = pyro.random_module("module", self, priors)
    lifted_reg_model = lifted_module()

    with pyro.plate("data", len(x_data)):
      logits = lifted_reg_model(x_data)
      obs = pyro.sample("obs", Categorical(logits=logits), obs=y_data)

    return logits

  def guide(self, x_data, y_data = None):
    # Variable for the softplus function
    softplus = torch.nn.Softplus()
    # Weight and bias variational distribution priors
    # First layer weight distribution priors
    fc1w_mu = torch.randn_like(self.fc1.weight)
    fc1w_sigma = torch.randn_like(self.fc1.weight)
    fc1w_mu_param = pyro.param("fc1w_mu", fc1w_mu)
    fc1w_sigma_param = softplus(pyro.param("fc1w_sigma", fc1w_sigma))
    fc1w_prior = Normal(loc=fc1w_mu_param, scale=fc1w_sigma_param).to_event(1)

    # First layer bias distribution priors
    fc1b_mu = torch.randn_like(self.fc1.bias)
    fc1b_sigma = torch.randn_like(self.fc1.bias)
    fc1b_mu_param = pyro.param("fc1b_mu", fc1b_mu)
    fc1b_sigma_param = softplus(pyro.param("fc1b_sigma", fc1b_sigma))
    fc1b_prior = Normal(loc=fc1b_mu_param, scale=fc1b_sigma_param).to_event(1)

    # Second layer weight distribution priors
    fc2w_mu = torch.randn_like(self.fc2.weight)
    fc2w_sigma = torch.randn_like(self,fc2.weight)
    fc2w_mu_param = pyro.param("fc2w_mu", fc2w_mu)
    fc2w_sigma_param = softplus(pyro.param("fc2w_sigma", fc2w_sigma))
    fc2w_prior = Normal(loc=fc2w_mu_param, scale=fc2w_sigma_param).to_event(1)

    # Second layer bias distribution priors
    fc2b_mu = torch.randn_like(self.fc2.bias)
    fc2b_sigma = torch.randn_like(self,fc2.bias)
    fc2b_mu_param = pyro.param("fc2b_mu", fc2b_mu)
    fc2b_sigma_param = softplus(pyro.param("fc2b_sigma", fc2b_sigma))
    fc2b_prior = Normal(loc=fc2b_mu_param, scale=fc2b_sigma_param).to_event(1)

    # Output layer weight distribution priors
    outputw_mu = torch.randn_like(self.out.weight)
    outputw_sigma = torch.randn_like(self.out.weight)
    outputw_mu_param = pyro.param("outputw_mu", outputw_mu)
    outputw_sigma_param = softplus(pyro.param("outputw_sigma", outputw_sigma))
    outputw_prior = Normal(loc=outputw_mu_param, scale=outputw_sigma_param).to_event(1)
    
    # Output layer bias distribution priors
    outputb_mu = torch.randn_like(self.out.bias)
    outputb_sigma = torch.randn_like(self.out.bias)
    outputb_mu_param = pyro.param("outputb_mu", outputb_mu)
    outputb_sigma_param = softplus(pyro.param("outputb_sigma", outputb_sigma))
    outputb_prior = Normal(loc=outputb_mu_param, scale=outputb_sigma_param).to_event(1)

    priors = {'fc1.weight': fc1w_prior, 'fc1.bias': fc1b_prior, 'fc2.weight': fc2w_prior, 'fc2.bias': fc2b_prior, 'output.weight': outputw_prior, 'output.bias': outputb_prior}
    lifted_module = pyro.random_module("module", self, priors)

    return lifted_module()

# Dataset loading block
All the lables are gathered and the dataset is loaded. It is then splitted into a train_set, validation_set and a test_set

In [5]:
# Define the list of file paths
file_path = './Chicks_Automatic_Detection_dataset/dt_Registrazioni_modificato/audio_spectrograms'
file_paths = [os.path.join(file_path, f) for f in os.listdir(file_path) if f.endswith('.pt')]

# Encode labels as integers
label_encoder = LabelEncoder()
all_labels = []

# First, gather all the labels to fit the encoder
for fp in file_paths:
    data = torch.load(fp)  # Use torch.load instead of pickle.load
    _, label = data['spectrogram'], data['label']
    all_labels.append(label)

label_encoder.fit(all_labels)

class SpectrogramDataset(Dataset):
    def __init__(self, file_paths, label_encoder):
        self.file_paths = file_paths
        self.label_encoder = label_encoder

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx])
        spectrogram, label = data['spectrogram'], data['label']
        # Encode the label as an integer
        encoded_label = self.label_encoder.transform([label])[0]
        return spectrogram, torch.tensor(encoded_label, dtype=torch.long)

# Actually loading the dataset
dataset = SpectrogramDataset(file_paths, label_encoder)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

# Splittong the data into train, test and validation sets
train_size = int(0.7*len(dataset))
validation_size = int(0.15*len(dataset))
test_size = len(dataset) - train_size - validation_size
train_dataset, test_dataset, validation_dataset = random_split(dataset, [train_size, test_size, validation_size])

# Creating the data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
validation_loader = DataLoader(validation_dataset, batch_size=32, shuffle=False)

FileNotFoundError: [Errno 2] No such file or directory: './Chicks_Automatic_Detection_dataset/dt_Registrazioni_modificato/audio_spectrograms'

# Training and validation phase

In [9]:
# Function to log values to a text file after each epoch
def log_epoch_data(epoch, avg_epoch_loss, avg_epoch_accuracy, avg_val_loss, avg_val_accuracy, model, filename='epoch_logs.txt'):
    with open(filename, 'a') as f:
        f.write(f"Epoch {epoch+1}:\n")
        f.write(f"Train Loss: {avg_epoch_loss:.4f}, Train Accuracy: {avg_epoch_accuracy:.4f}\n")
        f.write(f"Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {avg_val_accuracy:.4f}\n")
        f.write("Weight and Bias Distributions:\n")
        
        # Log weight and bias distributions from Pyro parameters
        pyro_param_store = pyro.get_param_store()
        for name in pyro_param_store.get_all_param_names():
            param_value = pyro_param_store[name].detach().cpu().numpy()
            f.write(f"{name}: {param_value}\n")
        f.write("\n")

# Training step
print("Starting training")

# Model instantiation
# output_size = 3 which are the different types of chicks calls
# Load a sample spectrogram to determine the input size
sample_data = torch.load(file_paths[0])
sample_spectrogram = sample_data['spectrogram']
input_size = spectrogram.numel()  # Flatten the spectrogram into a single vector (numel gives the total number of elements)
BNN_model = BNN(input_size=input_size, hidden_size=256, output_size=3)

# Define the optimizer (before loading checkpoint)
optimizer = torch.optim.Adam(BNN_model.parameters(), lr=0.01)

# SVI setup
optim = Adam({"lr": 0.01})
svi = SVI(BNN_model.model, BNN_model.guide, optim, loss=Trace_ELBO())

# Load checkpoint if available
start_epoch, _, _ = load_checkpoint(BNN_model, optimizer)

num_epoch = 20 # Maybe more(?)

# Create a logs directory if it doesn't exist
if not os.path.exists('logs'):
    os.makedirs('logs')

# Define the log file name
log_file = os.path.join('logs', 'training_logs.txt')

def calculate_accuracy(predictions, labels):
    _, predicted = torch.max(predictions, 1)
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)
    return accuracy

for epoch in range(start_epoch, num_epoch):
    BNN_model.train()
    epoch_loss = 0.0
    epoch_accuracy = 0.0

    # Training loop
    for x_train, y_train in train_loader:
        epoch_loss += svi.step(x_train, y_train)
        predictions = BNN_model(x_train)
        epoch_accuracy += calculate_accuracy(predictions, y_train)
    
    # Averaging the loss for the epoch
    avg_epoch_loss = epoch_loss / len(train_loader.dataset)
    avg_epoch_accuracy = epoch_accuracy / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epoch}, Loss: {avg_epoch_loss:.4f}, Accuracy: {avg_epoch_accuracy:.4f}")

    # Validation step
    BNN_model.eval()
    validation_loss = 0.0
    validation_accuracy = 0.0

    with torch.no_grad():
        for x_val, y_val in validation_loader:
            val_loss = svi.evaluate_loss(x_val, y_val)
            validation_loss += val_loss
            predictions = BNN_model(x_val)
            validation_accuracy += calculate_accuracy(predictions, y_val)
    
    avg_val_loss = validation_loss/len(validation_loader.dataset)
    avg_val_accuracy = validation_accuracy / len(validation_loader)
    print(f"Epoch {epoch+1}, Validation Loss: {avg_val_loss:.4f}, Validation Accuracy: {avg_val_accuracy:.4f}")

    # Log the values and parameter distributions for this epoch
    log_epoch_data(epoch, avg_epoch_loss, avg_epoch_accuracy, avg_val_loss, avg_val_accuracy, BNN_model, filename=log_file)

    # Save checkpoint at the end of each epoch
    save_checkpoint(BNN_model, optim, epoch, avg_epoch_loss, avg_epoch_accuracy)
    
print("Training completed")

# Testing phase

In [ ]:
# Prediction step
print("prediction step")
# Setting the model to evaluation mode
BNN_model.eval()
test_loss = 0.0

with torch.no_grad():
    for x_test, y_test in test_loader:
        test_loss += svi.evaluate_loss(x_test, y_test)

avg_test_loss = test_loss / len(test_loader.dataset)
print(f'Test Loss: {avg_test_loss}')